In [0]:
%sql
SELECT
    min(total_amount),
    max(total_amount),
    avg(total_amount),
    COUNT(*) AS total,
    COUNT(total_amount) AS not_null_total_amount_values,
    COUNT(*) - COUNT(total_amount) AS null_total_amount_values,
    min(tpep_pickup_datetime),
    max(tpep_pickup_datetime),
    MIN((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60) AS min_duration_minutes,
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60) AS max_duration_minutes,
    AVG((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60) AS avg_duration_minutes,
    COUNT(passenger_count) not_null_passenger_count_values,
    COUNT(*) - COUNT(passenger_count) null_passenger_count_values
FROM ifood_case.trusted.ny_taxi_trip_yellow

min(total_amount),max(total_amount),avg(total_amount),total,not_null_total_amount_values,null_total_amount_values,min(tpep_pickup_datetime),max(tpep_pickup_datetime),min_duration_minutes,max_duration_minutes,avg_duration_minutes,not_null_passenger_count_values,null_passenger_count_values
-982.95,6304.9,27.83854981703999,16186386,16186386,0,2001-01-01T00:06:49.000Z,2023-09-05T18:20:48.000Z,-476.25,10029.183333333332,16.827744530702113,15757721,428665


In [0]:
%sql
WITH durations AS (
    SELECT
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60 AS duration_min, 'yellow' as trip_type
    FROM ifood_case.trusted.ny_taxi_trip_yellow
    UNION ALL 
    SELECT
        (unix_timestamp(lpep_dropoff_datetime) - unix_timestamp(lpep_pickup_datetime)) / 60 AS duration_min, 'green' as trip_type
    FROM ifood_case.trusted.ny_taxi_trip_green
)
SELECT
    CASE
        WHEN duration_min < 0                      THEN '0 - Negativa'
        WHEN duration_min = 0                      THEN '1 - Zerada'
        WHEN duration_min > 0 AND duration_min < 1 THEN '2 - Menos de 1 min'
        WHEN duration_min < 5                       THEN '3 - Entre 1 a 5 min'
        WHEN duration_min < 15                      THEN '4 - Entre 5 a 15 min'
        WHEN duration_min < 30                      THEN '5 - Entre 15 a 30 min'
        WHEN duration_min < 60                      THEN '6 - Entre 30 a 60 min'
        WHEN duration_min < 120                     THEN '7 - Entre 1 a 2 horas'
        WHEN duration_min < 1440                    THEN '8 - Entre 2 a 24 horas'
        ELSE                                             '9 - Mais de 24 horas'
    END AS faixa_duracao_viagem,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS porcentagem
FROM durations
--where trip_type = 'yellow'
GROUP BY faixa_duracao_viagem
ORDER BY 1;


faixa_duracao_viagem,total,porcentagem
0 - Negativa,795,0.0048
1 - Zerada,5801,0.0351
2 - Menos de 1 min,181710,1.0995
3 - Entre 1 a 5 min,1760067,10.6503
4 - Entre 5 a 15 min,8223540,49.7612
5 - Entre 15 a 30 min,4623024,27.9742
6 - Entre 30 a 60 min,1500765,9.0812
7 - Entre 1 a 2 horas,210138,1.2716
8 - Entre 2 a 24 horas,20082,0.1215
9 - Mais de 24 horas,94,0.0006


In [0]:
%sql
SELECT
    CASE
        WHEN passenger_count IS NULL THEN '0 - Null'
        WHEN passenger_count < 0     THEN '1 - Negativo'
        WHEN passenger_count = 0     THEN '2 - Zero'
        WHEN passenger_count <= 4     THEN '3 - 1 a 4'
        WHEN passenger_count <= 6    THEN '4 - 5 a 6'
        ELSE                              '4 - Acima de 6'
    END AS faixa_total_passageiros,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS porcentagem
from
(
select passenger_count,
       'yellow' as trip_type
FROM ifood_case.trusted.ny_taxi_trip_yellow
union all
select passenger_count,
       'green' as trip_type
from ifood_case.trusted.ny_taxi_trip_green)
--where trip_type = 'green'
GROUP BY 1
ORDER BY 1;

faixa_total_passageiros,total,porcentagem
0 - Null,451561,2.7324
2 - Zero,275888,1.6694
3 - 1 a 4,15430512,93.3710
4 - 5 a 6,367913,2.2263
4 - Acima de 6,142,0.0009
